# 🏗️ Notebook 1: Netflix — Requirements & Architecture

## 🛠️ Setup

```bash
cd 06-system-designs/netflix
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we're designing

A global video streaming service. Think Netflix: hundreds of millions of users, petabytes
of video, low startup latency worldwide, personalized recommendations.

If you've never thought about streaming before, here's the one-sentence version:
**most of the work is moving video bytes close to the user, not computing anything clever
at request time.** The design is dominated by that fact.

### Functional requirements (the happy path)
- Upload & publish a video (admin side) — transcoded into multiple resolutions.
- Browse and search the catalog.
- **Stream** a video with good startup time and minimal rebuffering.
- Personalized **recommendations** ("Because you watched…").
- Track **playback position** so users resume where they left off.

### Non-functional
- **Read-heavy**: 99% of traffic is *watching*, not *uploading*. Optimize for reads.
- **Low startup**: <2s from click to first frame.
- **High availability**: 99.99%+ for playback. A dead catalog is bad; a dead player is worse.
- **Global**: users in Asia should not pull bytes from US-East.


## Back-of-envelope — let's actually compute it

Instead of guessing, let's put the numbers in Python so you can tweak them.

In [ ]:
# Simple back-of-envelope calculator - change the inputs and re-run.
users = 200_000_000
peak_concurrent_pct = 0.15           # 15% of users watching at peak
avg_bitrate_mbps = 3.0               # mix of SD/HD/4K
catalog_titles = 20_000
avg_title_gb = 5                     # one mezzanine / master copy
renditions = 6                       # 240p,360p,480p,720p,1080p,4k

concurrent = users * peak_concurrent_pct
peak_egress_tbps = concurrent * avg_bitrate_mbps / 1_000_000  # Mbps -> Tbps
storage_tb = catalog_titles * avg_title_gb * renditions / 1024

print(f"Peak concurrent viewers: {concurrent:>15,.0f}")
print(f"Peak egress:             {peak_egress_tbps:>15,.1f} Tbps")
print(f"Catalog storage:         {storage_tb:>15,.0f} TB")

# A single large cloud data center tops out near ~10 Tbps of egress.
# 90 Tbps / 10 Tbps = 9 data centers' worth of bandwidth JUST for video.
# That is why a CDN is not optional - it IS the architecture.
dc_cap_tbps = 10
print(f"\nData centers needed (no CDN): {peak_egress_tbps / dc_cap_tbps:.1f}")


## Bad → Best: where do the bytes come from?

Let's compare three architectures. The "bad" one is a naive first draft; each step fixes
the previous pain point.

In [ ]:
# --- v1 (BAD): origin serves all video ---
# One region, one data center, streams video directly to users.
# Users in Tokyo pull bytes from us-east-1. Latency is awful, egress is capped.
v1_problems = [
    "Egress bottleneck (~10 Tbps cap vs 90 Tbps demand)",
    "Trans-ocean latency (~150ms RTT)",
    "One viral episode can DoS the origin",
    "Bandwidth cost: you pay cloud egress for every single byte",
]
for p in v1_problems: print(" -", p)

# --- v2 (BETTER): add a CDN in front of the origin ---
# Origin stores video; CDN caches popular chunks at the edge (POPs).
# Cache-hit rate is the whole game. Let's compute origin load for different hit rates.
def origin_tbps(total_tbps: float, hit_rate: float) -> float:
    return total_tbps * (1 - hit_rate)

print("\nWith CDN in front of origin:")
for h in (0.50, 0.90, 0.95, 0.99):
    print(f"  hit rate {h:.0%} -> origin needs {origin_tbps(90, h):5.1f} Tbps")

# At 95% hit rate (realistic for head-of-catalog content), origin only needs
# 4.5 Tbps - fits in one region comfortably.

# --- v3 (BEST): CDN + multi-region origin + ISP-embedded caches ---
# Netflix OpenConnect puts caching appliances INSIDE ISPs.
# Result: the popular 99% of bytes never leave the ISP network -> near-zero backbone cost.


### Why cache hit rate is so high for video

Because catalog popularity follows a Zipf / power-law distribution: a small number of
titles account for most views. Let's see that.

In [ ]:
import random
from collections import Counter

random.seed(0)
N_TITLES = 20_000
N_REQUESTS = 100_000

# Zipf: popularity proportional to 1/rank^s
s = 1.1
weights = [1 / ((i + 1) ** s) for i in range(N_TITLES)]
total = sum(weights)
probs = [w / total for w in weights]

requests = random.choices(range(N_TITLES), weights=probs, k=N_REQUESTS)
counts = Counter(requests)

top500 = {t for t, _ in counts.most_common(500)}
hits = sum(1 for r in requests if r in top500)
print(f"Top-500 titles account for {hits/N_REQUESTS:.1%} of requests")
print("-> A cache holding just 500 titles serves most traffic.")


## High-level architecture

```
  [User]
    | https
    v
  +--------------------+
  |   Edge CDN (POPs)  |<-- 99% of bytes served here
  +--------+-----------+
           | cache miss / metadata
           v
  +--------------------+   +-----------------+
  |   API Gateway      |-->|  Auth, Rate lim |
  +--------+-----------+   +-----------------+
           |
   +-------+-------+---------------+--------------+
   v       v       v               v              v
 Catalog  User   Playback       Recommendation  Watch-history
 Service  Svc    Svc (manifests)   Svc           Svc
   |       |       |                |              |
   v       v       v                v              v
 MySQL  MySQL   Object Store    ML feature   Cassandra
                (S3)            store + models
                   ^
                   | write after encoding
  +----------------+------------------+
  |     Encoding / transcoding        |  (batch jobs, ffmpeg workers)
  |  mezzanine -> HLS/DASH x bitrates |
  +-----------------------------------+
```

### Key responsibilities
- **CDN**: primary *bytes* delivery. Netflix has its own (OpenConnect). A from-scratch
  design might use Cloudflare / Akamai / CloudFront.
- **Manifest**: a tiny playlist (HLS `.m3u8` or DASH MPD) telling the player which chunks
  to fetch at which bitrate.
- **ABR (Adaptive Bit Rate)**: player picks quality based on measured bandwidth, re-deciding
  every chunk. We simulate this in notebook 3.
- **Recommendations** are *precomputed* offline; serving is just a cache lookup.

### Why split "control plane" from "data plane"?
- Control plane (catalog, auth, recs lookup) = small JSON, CPU-bound, needs consistency.
- Data plane (video bytes) = huge, cacheable, needs bandwidth.
These have opposite scaling profiles - keeping them separate lets each scale on its own.


## Takeaways
1. **Do the back-of-envelope first.** It immediately tells you "a CDN is mandatory."
2. **Bad → Best is about cache hit rate.** CDN + popularity skew → origin load shrinks 20x.
3. **Split control plane from data plane.** Never stream video through your JSON API.
